# 06 · SkewerJAX and fake_spectra

`skewerjax.comparison` re-implements fake_spectra's scheme on SkewerJAX's absorbers and runs fake_spectra itself:

* **matched mode**: `FAKE_SPECTRA_EXTRACTION` + `FAKE_SPECTRA_INTEGRATION` reproduce fake_spectra (top-hat spheres,
  its mass and b constants, its star-forming rule, 7-point chords, per-cell pixel average, linear detuning, `tautail`,
  one redshift with a linear Hubble flow);
* **default**: SkewerJAX's choices; every difference is one named switch, and the **effect chain** goes from one to the
  other one switch at a time;
* `FakeSpectraSource` gives fake_spectra's own spectra and per-cell inputs, **live** (fake_spectra installed; each run
  in its own process with `python -m skewerjax.comparison.driver full …`, cached) or from a **reference** HDF5 file.

The full set of checks is `scripts/validation/check_fake_spectra.py`. Runtime: ~1 min (live).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, TOL_BRIGHT as TOL, add_panel_letters
use_style()
%matplotlib inline
import tempfile
from skewerjax.core import LineBundle
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, ExtractionOptions, HI
from skewerjax.comparison import (FakeSpectraSource, SKEWER_SETS, FAKE_SPECTRA_EXTRACTION as EF, FAKE_SPECTRA_INTEGRATION as IF,
                                  DEFAULT_INTEGRATION, INTEGRATION_EFFECTS, EXTRACTION_EFFECTS, matched_tau, run_effects, effect_rows,
                                  spectral_metrics)
CACHE = tempfile.mkdtemp(prefix="fs_cache_")             # where live fake_spectra results go (or pass reference="file.h5")
FS = FakeSpectraSource(cache=CACHE, snap_base="/home/STORAGE/TNG50-4", snapnum=25)
print("fake_spectra:", FS.mode)
reader = ArepoHDF5Reader("/home/STORAGE/TNG50-4", 25)
lya = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
sk = SimulationSkewers.axis_aligned(reader, SKEWER_SETS["single"]["cofm"], SKEWER_SETS["single"]["axis"], 0.25)
v, dv = sk.spectral_grid(1.0)                            # fake_spectra's periodic pixel grid
fs = FS.run("single_matched", "single", products="tau,colden")
t_matched = matched_tau(sk, HI, lya, v, EF, IF)
t_default = sk.tau(HI, lya, v)                            # SkewerJAX default (exact frame, on its own period)
print({k: f"{x:.1e}" for k, x in spectral_metrics(t_matched, fs["tau"], dv).items()})

In [ ]:
F_fs, F_m, F_d = np.exp(-fs["tau"][0]), np.exp(-t_matched[0]), np.exp(-t_default[0])
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
axs[0].plot(v, F_fs, color=EC["fake_spectra"], **MS["reference"], label="fake_spectra")
axs[0].plot(v, F_m, color=EC["HI"], **MS["matched"], label="SkewerJAX, matched")
axs[0].plot(v, F_d, color=EC["HI"], **MS["default"], alpha=0.5, label="SkewerJAX, default")
axs[1].plot(v, F_m - F_fs, color=EC["HI"], **MS["matched"], label="matched − fake_spectra")
axs[1].plot(v, F_d - F_fs, color=EC["HI"], **MS["default"], alpha=0.5, label="default − fake_spectra")
axs[0].set(ylabel="F", xlim=(1500, 2500)); axs[1].set(xlabel="u [km/s]", ylabel=r"$\Delta F$", ylim=(-1, 1))
axs[1].set_yscale("symlog", linthresh=1e-5); axs[1].set_yticks([-1e-1, -1e-3, -1e-5, 0, 1e-5, 1e-3, 1e-1])
for ax in axs: ax.legend(fontsize=13)
add_panel_letters(axs); fig.suptitle("One TNG50-4 skewer: matched mode reproduces fake_spectra (|ΔF| ~ 10⁻⁵); the default differs", fontsize=21); plt.show()

## Which differences matter?

The chain switches fake_spectra's options to SkewerJAX's one at a time (integration on fake_spectra's fields, then
the extraction); "alone" applies each switch by itself. The chain telescopes exactly, so the bars add up to the
total difference.

In [ ]:
res_i = run_effects(sk, HI, lya, v, INTEGRATION_EFFECTS, EF, IF, verbose=False)
res_e = run_effects(sk, HI, lya, v, EXTRACTION_EFFECTS, EF, DEFAULT_INTEGRATION, verbose=False)
rows = effect_rows(res_i, dv)[0] + effect_rows(res_e, dv)[0]
fig, ax = plt.subplots(figsize=(15, 7), layout="constrained")
y = np.arange(len(rows))
ax.barh(y + 0.2, [r["dF_chain"] for r in rows], 0.4, color=TOL["blue"], label="along the chain")
ax.barh(y - 0.2, [r["dF_iso"] for r in rows], 0.4, color=TOL["yellow"], label="alone")
ax.set_yticks(y, [r["effect"] for r in rows]); ax.axvline(0, color="k", lw=1)
ax.set_xscale("symlog", linthresh=1e-6); ax.set_xlabel(r"$\Delta\langle F\rangle$ of each switch (symmetric log scale)"); ax.legend()
fig.suptitle(f"Mean flux from fake_spectra to SkewerJAX: {sum(r['dF_chain'] for r in rows):+.4f} in total", fontsize=21); plt.show()

Each switch is an isolated option (`ExtractionOptions`, `IntegrationOptions`, `comparison.MatchedIntegration`), so any
combination can be evaluated with `matched_tau(sk, ion, lines, v, ext, integ)`. To use the comparison without
fake_spectra installed, export a reference file once in live mode (`FS.export("fake_spectra_reference.h5")`, or
`check_fake_spectra.py --export FILE`) and pass `FakeSpectraSource(reference=...)`.